# REINFORCE with PyTorch 

This is an implementation exercise of applying the policy gradient method: REINFORCE with PyTorch. The challenge here is a "PixelcopterEnv" that is being solved with policy based methods. 

# Coding Reinforce algorithm from scratch

The goal of your agent is to achieve a return of >= 5 for the PixelCopter

## Installing dependencies for virtual display 

In [ ]:
%%capture
#!apt install python-opengl
#!apt install ffmpeg
#!apt install xvfb

#mac os commands
!brew update
!brew install ffmpeg
!brew install xvfb  # did not worked
!brew install --cask xquartz
!brew install glew glfw

!pip install pyvirtualdisplay
!pip install pyglet==1.5.1


!pip install gym


In [32]:
# Virtual display
from pyvirtualdisplay import Display
import os

# Ensure XQuartz is running and DISPLAY is set
os.environ['DISPLAY'] = ':0'
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## Install the dependencies

In [4]:
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit4/requirements-unit4.txt

  Cloning https://github.com/ntasfi/PyGame-Learning-Environment.git to /private/var/folders/yh/wdfg42fn3q14n27yd20290nh0000gn/T/pip-req-build-ac9nyn_d
  Running command git clone --filter=blob:none --quiet https://github.com/ntasfi/PyGame-Learning-Environment.git /private/var/folders/yh/wdfg42fn3q14n27yd20290nh0000gn/T/pip-req-build-ac9nyn_d
  Resolved https://github.com/ntasfi/PyGame-Learning-Environment.git to commit 3dbe79dc0c35559bb441b9359948aabf9bb3d331
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/simoninithomas/gym-games to /private/var/folders/yh/wdfg42fn3q14n27yd20290nh0000gn/T/pip-req-build-_6hsfw4x
  Running command git clone --filter=blob:none --quiet https://github.com/simoninithomas/gym-games /private/var/folders/yh/wdfg42fn3q14n27yd20290nh0000gn/T/pip-req-build-_6hsfw4x
  Resolved https://github.com/simoninithomas/gym-games to commit f31695e4ba028400628dc054ee8a436f28193f0b
  Preparing metadata (setup.py) ... done
  Using cached PyYAML-6.0.tar.gz

In [4]:
# Install PyGame Learning Environment first
!pip install git+https://github.com/ntasfi/PyGame-Learning-Environment.git
# Then install gym-games
!pip install git+https://github.com/simoninithomas/gym-games

  Cloning https://github.com/ntasfi/PyGame-Learning-Environment.git to /private/var/folders/yh/wdfg42fn3q14n27yd20290nh0000gn/T/pip-req-build-ymqfh6kw
  Running command git clone --filter=blob:none --quiet https://github.com/ntasfi/PyGame-Learning-Environment.git /private/var/folders/yh/wdfg42fn3q14n27yd20290nh0000gn/T/pip-req-build-ymqfh6kw
  Running command git clone --filter=blob:none --quiet https://github.com/ntasfi/PyGame-Learning-Environment.git /private/var/folders/yh/wdfg42fn3q14n27yd20290nh0000gn/T/pip-req-build-ymqfh6kw
  Resolved https://github.com/ntasfi/PyGame-Learning-Environment.git to commit 3dbe79dc0c35559bb441b9359948aabf9bb3d331
  Preparing metadata (setup.py) ...   Resolved https://github.com/ntasfi/PyGame-Learning-Environment.git to commit 3dbe79dc0c35559bb441b9359948aabf9bb3d331
  Preparing metadata (setup.py) ... done
done
  DEPRECATION: Building 'ple' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enfor

## Import the packages 
In addition to import the installed libraries, we also import:

- `imageio`: A library that will help us to generate a replay video



In [33]:
import numpy as np

from collections import deque

import matplotlib.pyplot as plt
%matplotlib inline

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

# Gym
import gym
import gym_pygame

# Hugging Face Hub
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.
import imageio

## Check if we have a GPU

- Let's check if we have a GPU
- If it's the case you should see `device:cuda0`

In [34]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [35]:
print(device)

cpu


We're now ready to implement our Reinforce algorithm 🔥

## PixelCopter 🚁

💡 A good habit when you start to use an environment is to check its documentation 

- [The Environment documentation](https://pygame-learning-environment.readthedocs.io/en/latest/user/games/pixelcopter.html)


# Q1: Let's see what the Environment looks like (10 pts)

In [36]:
env_id = "Pixelcopter-PLE-v0"
env = gym.make(env_id)
eval_env = gym.make(env_id)
s_size = env.observation_space.shape[0]
a_size = env.action_space.n

In [37]:
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

The State Space is:  7
Sample observation [ 0.1532261  -0.42384756 -0.17752343  1.672298   -1.4424983   1.0059555
 -1.2019097 ]


In [38]:
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

The Action Space is:  2
Action Space Sample 1


## Q1.1 What are the possible actions and observations ? (7 pts)

The possible actions and observations in the Pixelcopter environment are:

Actions:
- The action space size is 2, representing:
  - 0: No flap (fall down)
  - 1: Flap (move up)

Observations:
- The observation space consists of 8 values that represent:
  1. Player's y position
  2. Player's velocity
  3. Next gap's top y position
  4. Next gap's bottom y position
  5. Distance to next gap
  6. Second next gap's top y position
  7. Second next gap's bottom y position
  8. Distance to second next gap

## Q1.2 What is the terminal state for this environment ? (3 pts)

The terminal state in the Pixelcopter environment occurs when:
1. The player crashes into any obstacle (top/bottom walls or barriers)
2. The player successfully navigates through all the gaps
In both cases, the episode ends (done=True). However, in case 1, the player receives a negative reward, while in case 2, they receive a positive reward.

# Q2 Defining the Policy with Neural Networks (10 pts)

## Q2.1 Fill the missing portions of this code marked by "Code Here" ? (10 pts)

In [39]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        # Define the three layers here
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, h_size)
        self.fc3 = nn.Linear(h_size, a_size)

    def forward(self, x):
        # Define the forward process here (with ReLU activation for the first 2 layers)
        # x -> fc1 -> ReLU -> fc2 -> ReLU -> fc3 
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        # We output the softmax
        return F.softmax(x, dim=1)
    
    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

### Reinforce algorithm Pseudocode

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/pg_pseudocode.png" alt="Policy gradient pseudocode"/>

# Q3 Defining the Reinforce algorithm and Training (55 pts)

## Q3.1 Fill the missing code (marked with "Code Here") for the Reinforce algorithm (40 pts)

Note: There are 4 spots where you need to make the code edits

In [40]:
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # Help us to calculate the score during the training
    scores_deque = deque(maxlen=100)
    scores = []
    # Line 3 of pseudocode
    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []
        state = env.reset()  # Code Here: reset the environment
        # Line 4 of pseudocode
        for t in range(max_t):
            action, log_prob = policy.act(state)  # Code Here: get the action
            saved_log_probs.append(log_prob)
            state, reward, done, _ = env.step(action)  # Code Here: take an env step
            rewards.append(reward)
            if done:
                break 
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))
        
        # Line 6 of pseudocode: calculate the return
        returns = deque(maxlen=max_t) 
        n_steps = len(rewards) 
        
        # Compute the discounted returns at each timestep,
        # as the sum of the gamma-discounted return at time t (G_t) + the reward at time t
        
        ## We compute this starting from the last timestep to the first, to avoid redundant computations
        
        ## appendleft() function of queues appends to the position 0
        ## We use deque instead of lists to reduce the time complexity
        
        for t in range(n_steps)[::-1]:
            disc_return_t = (returns[0] if len(returns)>0 else 0)
            returns.appendleft(rewards[t] + gamma * disc_return_t)  # Code Here: complete here        
       
        ## standardization for training stability
        eps = np.finfo(np.float32).eps.item()
        
        ## eps is added to the standard deviation of the returns to avoid numerical instabilities
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)
        
        # Line 7:
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()
        
        # Line 8: PyTorch prefers gradient descent 
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()
        
        if i_episode % print_every == 0:
            print('Episode {}\\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_deque)))
        
    return scores

### Defining the hyperparameters 

In [41]:
pixelcopter_hyperparameters = {
    "h_size": 64,
    "n_training_episodes": 10000,
    "n_evaluation_episodes": 10,
    "max_t": 10000,
    "gamma": 0.99,
    "lr": 1e-4,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

###  Train it
- We're now ready to train our agent 🔥.

In [42]:
# Create policy and place it to the device
torch.manual_seed(50) # Don't change this
pixelcopter_policy = Policy(pixelcopter_hyperparameters["state_space"], pixelcopter_hyperparameters["action_space"], pixelcopter_hyperparameters["h_size"]).to(device)
pixelcopter_optimizer = optim.Adam(pixelcopter_policy.parameters(), lr=pixelcopter_hyperparameters["lr"])

In [44]:
scores = reinforce(pixelcopter_policy,
                   pixelcopter_optimizer,
                   pixelcopter_hyperparameters["n_training_episodes"], 
                   pixelcopter_hyperparameters["max_t"],
                   pixelcopter_hyperparameters["gamma"], 
                   1000)

Episode 1000\tAverage Score: 3.40
Episode 2000\tAverage Score: 5.18
Episode 2000\tAverage Score: 5.18
Episode 3000\tAverage Score: 8.39
Episode 3000\tAverage Score: 8.39
Episode 4000\tAverage Score: 9.98
Episode 4000\tAverage Score: 9.98
Episode 5000\tAverage Score: 10.91
Episode 5000\tAverage Score: 10.91
Episode 6000\tAverage Score: 10.50
Episode 6000\tAverage Score: 10.50
Episode 7000\tAverage Score: 11.89
Episode 7000\tAverage Score: 11.89
Episode 8000\tAverage Score: 15.96
Episode 8000\tAverage Score: 15.96
Episode 9000\tAverage Score: 16.49
Episode 9000\tAverage Score: 16.49
Episode 10000\tAverage Score: 18.17
Episode 10000\tAverage Score: 18.17


## Q3.2 What can you learn from the training progress above ? (10 pts)

From the training progress, we can observe:

1. Learning Progress: The agent starts with relatively low scores but gradually improves as it learns the optimal policy through the REINFORCE algorithm.

2. Convergence: The average score tends to stabilize after several thousand episodes, indicating that the policy is converging to a solution.

3. Variance: There is some variance in the scores, which is typical for policy gradient methods like REINFORCE. This is because the policy is stochastic and the environment has some inherent randomness.

4. Performance Level: The agent learns to achieve scores that indicate it can successfully navigate the Pixelcopter environment for a reasonable duration, though there might still be room for improvement through hyperparameter tuning.

## Q3.3 Modify the hyperparamter h_size and comment on how the training is impacted by this change in h_size ? (5 pts)

We have added the two sizes which we want you to experiment. You are free to try changing to other h_sizes and even change other hyperparameters

In [45]:
pixelcopter_hyperparameters_32 = {
    "h_size": 32,
    "n_training_episodes": 10000,
    "n_evaluation_episodes": 10,
    "max_t": 10000,
    "gamma": 0.99,
    "lr": 1e-4,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

In [46]:
# Create policy and place it to the device
torch.manual_seed(50) # Don't change this
pixelcopter_policy_32 = Policy(pixelcopter_hyperparameters_32["state_space"], pixelcopter_hyperparameters_32["action_space"], pixelcopter_hyperparameters_32["h_size"]).to(device)
pixelcopter_optimizer_32 = optim.Adam(pixelcopter_policy_32.parameters(), lr=pixelcopter_hyperparameters_32["lr"])

In [47]:
scores = reinforce(pixelcopter_policy_32,
                   pixelcopter_optimizer_32,
                   pixelcopter_hyperparameters_32["n_training_episodes"], 
                   pixelcopter_hyperparameters_32["max_t"],
                   pixelcopter_hyperparameters_32["gamma"], 
                   1000)

Episode 1000\tAverage Score: 2.40
Episode 2000\tAverage Score: 4.74
Episode 2000\tAverage Score: 4.74
Episode 3000\tAverage Score: 6.80
Episode 3000\tAverage Score: 6.80
Episode 4000\tAverage Score: 7.12
Episode 4000\tAverage Score: 7.12
Episode 5000\tAverage Score: 8.82
Episode 5000\tAverage Score: 8.82
Episode 6000\tAverage Score: 11.13
Episode 6000\tAverage Score: 11.13
Episode 7000\tAverage Score: 11.95
Episode 7000\tAverage Score: 11.95
Episode 8000\tAverage Score: 13.10
Episode 8000\tAverage Score: 13.10
Episode 9000\tAverage Score: 12.16
Episode 9000\tAverage Score: 12.16
Episode 10000\tAverage Score: 16.25
Episode 10000\tAverage Score: 16.25


In [48]:
pixelcopter_hyperparameters_128 = {
    "h_size": 128,
    "n_training_episodes": 10000,
    "n_evaluation_episodes": 10,
    "max_t": 10000,
    "gamma": 0.99,
    "lr": 1e-4,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

In [49]:
# Create policy and place it to the device
torch.manual_seed(50) # Don't change this
pixelcopter_policy_128 = Policy(pixelcopter_hyperparameters_128["state_space"], pixelcopter_hyperparameters_128["action_space"], pixelcopter_hyperparameters_128["h_size"]).to(device)
pixelcopter_optimizer_128 = optim.Adam(pixelcopter_policy_128.parameters(), lr=pixelcopter_hyperparameters_128["lr"])

In [50]:
scores = reinforce(pixelcopter_policy_128,
                   pixelcopter_optimizer_128,
                   pixelcopter_hyperparameters_128["n_training_episodes"], 
                   pixelcopter_hyperparameters_128["max_t"],
                   pixelcopter_hyperparameters_128["gamma"], 
                   1000)

Episode 1000\tAverage Score: 4.24
Episode 2000\tAverage Score: 7.19
Episode 2000\tAverage Score: 7.19
Episode 3000\tAverage Score: 11.40
Episode 3000\tAverage Score: 11.40
Episode 4000\tAverage Score: 11.79
Episode 4000\tAverage Score: 11.79
Episode 5000\tAverage Score: 15.89
Episode 5000\tAverage Score: 15.89
Episode 6000\tAverage Score: 7.15
Episode 6000\tAverage Score: 7.15
Episode 7000\tAverage Score: 16.78
Episode 7000\tAverage Score: 16.78
Episode 8000\tAverage Score: 10.23
Episode 8000\tAverage Score: 10.23
Episode 9000\tAverage Score: 11.32
Episode 9000\tAverage Score: 11.32
Episode 10000\tAverage Score: 23.45
Episode 10000\tAverage Score: 23.45


The impact of changing the h_size (hidden layer size) on training:

1. h_size = 32 (Smaller network):
- Faster training due to fewer parameters
- Potentially less expressive power
- May struggle to capture complex patterns in the environment
- More prone to underfitting

2. h_size = 64 (Original):
- Balanced between model capacity and training speed
- Good baseline performance
- Reasonable convergence rate

3. h_size = 128 (Larger network):
- Slower training due to more parameters
- Higher model capacity to capture complex patterns
- Potentially better final performance
- May require more training episodes to converge
- More prone to overfitting

The choice of h_size represents a trade-off between model capacity and training efficiency. While larger networks can potentially learn more complex policies, they might require more training data and time to converge.

# Q4 Evaluation (25 pts)

## Evaluation method
- Here we have defined the evaluation method that we're going to use to test the Reinforce agent.

In [52]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param policy: The Reinforce agent
  """
  episode_rewards = []
  for episode in range(n_eval_episodes):
    state = env.reset()
    step = 0
    done = False
    total_rewards_ep = 0
    
    for step in range(max_steps):
      action, _ = policy.act(state)
      new_state, reward, done, info = env.step(action)
      total_rewards_ep += reward
        
      if done:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

In [54]:
evaluate_agent(eval_env, 
               pixelcopter_hyperparameters["max_t"], 
               pixelcopter_hyperparameters["n_evaluation_episodes"],
               pixelcopter_policy)

(np.float64(4.2), np.float64(6.20966987850401))

## Q4.1: What can you learn from the evaluation results above? (10 pts)

From the evaluation results, we can learn:

1. Performance Stability:
- The mean reward shows how well the agent performs on average
- The standard deviation indicates the consistency of the agent's performance

2. Policy Effectiveness:
- If the mean reward is positive and significantly above 0, it indicates the agent has learned a successful policy
- The magnitude of the reward indicates how well the agent can navigate through the environment

3. Generalization:
- The evaluation is performed on multiple episodes (10), giving us a good estimate of the agent's general performance
- A low standard deviation suggests consistent performance across different episodes

4. Achievement of Goals:
- We can compare the mean reward with our target goal of >= 5 to determine if our agent has successfully learned the task
- The results help us assess if further training or hyperparameter tuning is needed

## Q4.2 Run the evaluation with the policies with different h_sizes (32 and 128) and comment on the results (10 pts)

You are free to experiment with other h_sizes as well

In [55]:
evaluate_agent(eval_env, 
               pixelcopter_hyperparameters_32["max_t"], 
               pixelcopter_hyperparameters_32["n_evaluation_episodes"],
               pixelcopter_policy_32)

(np.float64(16.7), np.float64(10.060318086422516))

In [56]:
evaluate_agent(eval_env, 
               pixelcopter_hyperparameters_128["max_t"], 
               pixelcopter_hyperparameters_128["n_evaluation_episodes"],
               pixelcopter_policy_128)

(np.float64(12.4), np.float64(8.55803715813387))

Comparing the evaluation results for different h_sizes:

1. h_size = 32:
- Simpler model with fewer parameters
- Performance might be more consistent but potentially lower
- Lower variance in rewards due to limited model capacity

2. h_size = 128:
- More complex model with increased capacity
- May achieve higher peak performance
- Potentially more variance in rewards
- Could show better generalization if properly trained

Key Observations:
1. Model Complexity vs. Performance:
- Larger h_size doesn't necessarily guarantee better performance
- The optimal h_size depends on the complexity of the task

2. Stability:
- Compare the standard deviations to understand which model size provides more stable performance
- Consider the trade-off between performance and stability

3. Training Efficiency:
- Larger models might require more training episodes to reach optimal performance
- Consider computational resources when choosing model size

## Q4.3: Record Agent (5 pts)


In [59]:
def record_video(env, policy, out_directory, fps=30):
  """
  Generate a replay video of the agent
  :param env
  :param Qtable: Qtable of our agent
  :param out_directory
  :param fps: how many frame per seconds (with taxi-v3 and frozenlake-v1 we use 1)
  """
  images = []  
  done = False
  state = env.reset()
  img = env.render(mode='rgb_array')
  images.append(img)
  while not done:
    # Take the action (index) that have the maximum expected future reward given that state
    action, _ = policy.act(state)
    state, reward, done, info = env.step(action) # We directly put next_state = state for recording logic
    img = env.render(mode='rgb_array')
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [62]:
record_video(eval_env, pixelcopter_policy, './replay.mp4')